# Notebook 15: Margin, collateral, and XVA

**Prerequisites:** Work through the foundation track first — notebooks **01** (core types and money), **02** (dates, calendars, schedules), **03** (market data and curves), and **04** (math toolkit). Those notebooks introduce currencies, dates, and basic finstack patterns used implicitly here.

This notebook introduces the `finstack.margin` API: netting sets, CSA specs, variation margin (VM), eligible collateral schedules, utilization metrics, and configuration types that feed XVA-style workflows.

**Scope note:** This notebook focuses on **margin and collateral configuration** (netting sets, CSA, VM, eligible collateral, utilization, and `FundingConfig` / `XvaConfig` as configuration types). A full **CVA** walkthrough (exposure profiles, netting, and discounting) is intentionally out of scope here and could be added later as a separate pedagogical section.

## Concepts: margin, collateral, and XVA

**Variation margin (VM)** is the bilateral exchange of collateral driven by mark-to-market exposure under a **Credit Support Annex (CSA)**. Thresholds, independent amounts, and minimum transfer amounts determine when a **margin call** is required and how much must be delivered or returned.

**Collateral eligibility** (cash vs. securities, haircuts, rehypothecation) is captured in schedules such as BCBS-style baskets or cash-only setups.

**XVA** (collectively CVA, DVA, FVA, etc.) adjusts derivative values for credit, funding, and related effects. Full XVA engines simulate exposure paths over time; here we focus on **configuration types** — for example `FundingConfig` encodes funding spread and benefit assumptions used when pricing funding costs (FVA) or related adjustments.

The API below is **self-contained**: identifiers, CSA presets, VM calculation, collateral schedules, utilization, and XVA/funding configuration.

## API walkthrough: imports and availability

We import the main margin types from `finstack.margin`. If the extension module is missing (e.g. wheel not built), we record the error so later cells can skip gracefully while still printing something.

In [ ]:
try:
    from finstack.margin import (
        NettingSetId,
        CsaSpec,
        VmCalculator,
        XvaConfig,
        FundingConfig,
        MarginUtilization,
        EligibleCollateralSchedule,
    )

    MARGIN_AVAILABLE = True
    _IMPORT_ERROR = None
except Exception as e:
    MARGIN_AVAILABLE = False
    _IMPORT_ERROR = e
    NettingSetId = CsaSpec = VmCalculator = None  # type: ignore[misc, assignment]
    XvaConfig = FundingConfig = MarginUtilization = EligibleCollateralSchedule = None  # type: ignore[misc, assignment]

if MARGIN_AVAILABLE:
    print("finstack.margin: imports OK")
else:
    print(f"finstack.margin: import failed — {_IMPORT_ERROR!r}")

### Netting set identifiers

A **netting set** groups trades for which close-out and margin apply together. **Bilateral** sets are keyed by counterparty and CSA id; **cleared** sets are keyed by CCP id.

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    ns_bilateral = NettingSetId.bilateral("CPTY-1", "CSA-001")
    ns_cleared = NettingSetId.cleared("LCH")
    print(f"Bilateral: {ns_bilateral}")
    print(f"Cleared: {ns_cleared}")

### `CsaSpec`: regulatory-style CSA presets

`CsaSpec` wraps the legal/economic terms needed for VM logic. Factory methods such as `usd_regulatory()` and `eur_regulatory()` provide ready-made specs; `to_json()` supports serialization.

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    csa = CsaSpec.usd_regulatory()
    print(f"CSA: {csa}")
    print(f"JSON: {csa.to_json()}")
    csa_eur = CsaSpec.eur_regulatory()
    print(f"EUR CSA: {csa_eur}")

### `VmCalculator`: variation margin from exposure and posted collateral

Given mark-to-market **exposure**, **posted collateral**, **currency**, and a **valuation date** (`year`, `month`, `day`), the calculator returns gross/net exposure, delivery and return amounts, net margin, and whether a **margin call** is required.

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    vm = VmCalculator(CsaSpec.usd_regulatory())
    result = vm.calculate(1_000_000.0, 0.0, "USD", 2025, 1, 15)
    print(f"Gross exposure: {result.gross_exposure:,.2f}")
    print(f"Net exposure: {result.net_exposure:,.2f}")
    print(f"Delivery amount: {result.delivery_amount:,.2f}")
    print(f"Return amount: {result.return_amount:,.2f}")
    print(f"Net margin: {result.net_margin:,.2f}")
    print(f"Requires call: {result.requires_call}")
    result2 = vm.calculate(-500_000.0, 200_000.0, "USD", 2025, 1, 15)
    print(
        f"\nNeg exposure: gross={result2.gross_exposure:,.2f}, net={result2.net_exposure:,.2f}"
    )
    print(f"Requires call: {result2.requires_call}")

### `EligibleCollateralSchedule`: what can be posted

Schedules describe which asset classes are eligible and with what haircuts — from **cash-only** to **BCBS-standard** baskets or **US Treasury** schedules.

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    cash = EligibleCollateralSchedule.cash_only()
    print(f"Cash only: {cash}")
    bcbs = EligibleCollateralSchedule.bcbs_standard()
    print(f"BCBS standard: {bcbs}")
    ust = EligibleCollateralSchedule.us_treasuries()
    print(f"US Treasuries: {ust}")

### `MarginUtilization`: posted vs. required

`MarginUtilization(posted_amount, required_amount, currency)` summarizes how fully margin covers the requirement (ratio, adequacy, shortfall).

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    util = MarginUtilization(1_200_000.0, 1_000_000.0, "USD")
    print(f"Utilization: {util}")
    print(f"Posted: {util.posted:,.2f}, required: {util.required:,.2f}, ratio: {util.ratio:.4f}")
    print(f"Adequate: {util.is_adequate()}, shortfall: {util.shortfall():,.2f}")

### `FundingConfig` and `XvaConfig`

`FundingConfig(funding_spread_bps, funding_benefit_bps=None)` holds spreads used in funding-aware valuation. `XvaConfig()` builds a default time grid and recovery assumptions; it can optionally embed a `FundingConfig`.

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    funding = FundingConfig(50.0, 30.0)
    print(f"Funding: {funding}")
    print(
        f"spread_bps={funding.funding_spread_bps}, "
        f"benefit_bps={funding.funding_benefit_bps}, "
        f"effective_benefit_bps={funding.effective_benefit_bps()}"
    )
    xva_cfg = XvaConfig()
    print(f"XVA config: {xva_cfg}")
    print(f"Time grid length: {len(xva_cfg.time_grid)}, recovery_rate: {xva_cfg.recovery_rate}")

## Mini-example: bilateral CSA, VM, and funding cost

1. Use a **USD regulatory** CSA (typical bilateral VM context).
2. Portfolio **+$5M** exposure with **$1M** already posted.
3. Inspect **margin call** flags and amounts.
4. Flip to **negative exposure** and see how VM logic responds.
5. Tie **funding spreads** to a simple **annual funding cost** illustration on posted margin.

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    ns = NettingSetId.bilateral("ACME-DEALER", "CSA-USD-REG")
    csa_usd = CsaSpec.usd_regulatory()
    calc = VmCalculator(csa_usd)
    exposure_pos = 5_000_000.0
    posted = 1_000_000.0
    as_of = (2025, 4, 11)
    r = calc.calculate(exposure_pos, posted, "USD", *as_of)
    print(f"Netting set: {ns}")
    print(f"CSA: {csa_usd}")
    print(f"Exposure +$ {exposure_pos:,.0f}, posted $ {posted:,.0f}")
    print(f"Gross exposure: {r.gross_exposure:,.2f}")
    print(f"Net exposure: {r.net_exposure:,.2f}")
    print(f"Delivery: {r.delivery_amount:,.2f}, return: {r.return_amount:,.2f}")
    print(f"Net margin: {r.net_margin:,.2f}")
    print(f"Requires margin call: {r.requires_call}")
    u = MarginUtilization(posted, exposure_pos, "USD")
    print(f"Posted vs gross MTM (illustrative): {u}")
    print(f"  ratio={u.ratio:.4f}, adequate={u.is_adequate()}, shortfall={u.shortfall():,.2f}")

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    calc = VmCalculator(CsaSpec.usd_regulatory())
    exposure_neg = -5_000_000.0
    posted_still = 1_000_000.0
    r_rev = calc.calculate(exposure_neg, posted_still, "USD", 2025, 4, 11)
    print("Exposure reversed (we owe / negative MTM):")
    print(f"Gross exposure: {r_rev.gross_exposure:,.2f}")
    print(f"Net exposure: {r_rev.net_exposure:,.2f}")
    print(f"Delivery: {r_rev.delivery_amount:,.2f}, return: {r_rev.return_amount:,.2f}")
    print(f"Net margin: {r_rev.net_margin:,.2f}")
    print(f"Requires margin call: {r_rev.requires_call}")

In [ ]:
if not MARGIN_AVAILABLE:
    print("Skip: margin module not available")
else:
    funding = FundingConfig(50.0, 30.0)
    spread_bps = funding.funding_spread_bps
    notional_for_funding = 4_000_000.0
    annual_funding_cost = notional_for_funding * spread_bps / 10_000.0
    print(
        "Concept: annual funding cost on cash or collateral notionally tied to "
        f"${notional_for_funding:,.0f} at {spread_bps} bp spread"
    )
    print(f"  ≈ ${annual_funding_cost:,.2f} per year (simple notional × bps / 10_000)")
    print(f"FundingConfig repr: {funding}")
    xva_with_funding = XvaConfig(funding=funding)
    print(f"XvaConfig with embedded funding: {xva_with_funding}")

## Takeaways

- **NettingSetId** distinguishes bilateral (counterparty + CSA) from cleared (CCP) contexts; use consistent ids across risk and margin systems.
- **CsaSpec** drives **VmCalculator**: thresholds and CSA terms determine **delivery**, **return**, **net margin**, and **requires_call**.
- **EligibleCollateralSchedule** presets model what can be posted (cash, BCBS-style, Treasuries) for policy and haircut logic.
- **MarginUtilization** summarizes **posted vs. required** with ratio, adequacy, and shortfall.
- **FundingConfig** / **XvaConfig** connect margin and funding assumptions to XVA-style configuration; simple **notional × spread (bps) / 10,000** illustrates annual funding cost magnitude.

Next steps: combine this with pricing notebooks (e.g. **05–06**) to feed exposures into margin and, where available, full XVA engines.